In [0]:
service_credential = dbutils.secrets.get(scope='scope-test-learning-awg', key='azure-ms-entra-secret-key-vault-governance-awg')
 
spark.conf.set("fs.azure.account.auth.type.datalaketestlearningawg.dfs.core.windows.net", "OAuth")
spark.conf.set("fs.azure.account.oauth.provider.type.datalaketestlearningawg.dfs.core.windows.net", "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider")
spark.conf.set("fs.azure.account.oauth2.client.id.datalaketestlearningawg.dfs.core.windows.net", "504d82b4-c2c8-496f-9de9-b944b4f1b9f6")
spark.conf.set("fs.azure.account.oauth2.client.secret.datalaketestlearningawg.dfs.core.windows.net", service_credential)
spark.conf.set("fs.azure.account.oauth2.client.endpoint.datalaketestlearningawg.dfs.core.windows.net", "https://login.microsoftonline.com/50a3af05-f42e-4ad1-9203-9b4d0f7b4664/oauth2/token")

In [0]:
data = [(1,"aniket"), (2,"baddy"), (3, "sujit")]
schema = """id int, name string"""
df = spark.createDataFrame(data, schema)
df.show()

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.window import Window
from pyspark.sql.functions import window
schema = StructType([StructField("id", IntegerType(), nullable=False), StructField("name", StringType(), nullable=True)])
data = [(1,"aniket"), (2,"baddy"), (3, "sujit"), (None, None)]
df = spark.createDataFrame(data, schema)
df.show()

In [0]:
df = spark.read.format('csv').option("header", "true").option("inferSchema", "true").load("abfss://test-container3@datalaketestlearningawg.dfs.core.windows.net/testdir1/habit names.csv")
df.show()

In [0]:
from pyspark.sql.functions import col
df.filter((col("Habit Names").like("R%")) | (col("Habit Names").like("%s"))).show()

In [0]:
from pyspark.sql.functions import regexp_replace
df.withColumn("testregexp", regexp_replace(col("Habit Names"), "ss", "ssssssssssss")).show()

In [0]:
df = spark.read.format('csv').option("header", "true").option("inferSchema", "true").load("abfss://test-container3@datalaketestlearningawg.dfs.core.windows.net/testdir1/simple-zipcodes.csv")
df.show()

In [0]:
df.write.format('delta').mode('overwrite').saveAsTable('man_cata1.man_schema1.simple_zipcodes_tbl')
#tenant added - so old catalogs are in normal outlook account and permission not added for EXT account

In [0]:
from pyspark.sql.functions import sum
df.withColumn("testcol1", sum(col('RecordNumber')).over(Window.partitionBy('State'))).show()

In [0]:
df.withColumn("testcol1", sum(col('RecordNumber')).over(Window.partitionBy('State').orderBy(col('State').asc()))).show()

In [0]:
df.withColumn("testcol1", sum(col('RecordNumber')).over(Window.partitionBy('State').orderBy(col('RecordNumber').asc()))).show()

In [0]:
df.withColumn("testcol1", sum(col('RecordNumber')).over(Window.orderBy(col('State').asc()))).show()  #by defualt cumulative behaviour - rangeBetween(Window.unboundedPreceding,0)

In [0]:
df.withColumn("testcol1", sum(col('RecordNumber')).over(Window.orderBy(col('State').asc()).rangeBetween(Window.unboundedPreceding,0))).show() #range b/w recommends numberical/date column - but works on ony column type
# same answer as above query

In [0]:
df.withColumn("testcol1", sum(col('RecordNumber')).over(Window.orderBy(col('State').asc()).rowsBetween(Window.unboundedPreceding,0))).show()

In [0]:
df.join(df, on='State', how='inner').show(5)

In [0]:
df.join(df, on='State', how='left').show(5)

In [0]:
df.join(df, on='State', how='right').show(5)
# df.join(df, on='State', how='full').show(5)
# df.join(df, on='State', how='cross').show(5)

In [0]:
df.join(df, on='State', how='anti').show(5)

In [0]:
df1 = df.alias("df1")
df2 = df.alias("df2")

# Use qualified column names in join condition
df1.join(df2, df1['State'] == df2['State'], how='inner').select('df1.State').show()

# df.join(df, df['State'] == df['State'], how='left').show()
# df.join(df, df['State'] == df['State'], how='right').show()
# df.join(df, df['State'] == df['State'], how='full').show()
# df.join(df, df['State'] == df['State'], how='cross').show()
# df.join(df, df['State'] == df['State'], how='anti').show()

In [0]:
from pyspark.sql.functions import when
df.withColumn("State", when(col("State") == "AL", "ALHAZARRRR").when(col("State") == "PR", "PRRRRRRRR").otherwise(col("State"))).show()

In [0]:
from pyspark.sql.functions import sum, avg, collect_list
df.groupBy('State').agg(sum('RecordNumber').alias('summmm'), avg('RecordNumber').alias('avgggg')).show()
df.groupBy('State').agg(collect_list('City').alias('colectttt')).show()


In [0]:
df = spark.read.format('csv').option("header", "true").option("inferSchema", "true").load("abfss://test-container3@datalaketestlearningawg.dfs.core.windows.net/testdir1/sales.csv")
df.show()

In [0]:
from pyspark.sql.functions import to_timestamp, to_date, date_add, date_format, datediff, timestamp_add
df=df.withColumn('timestamp_col', to_timestamp(col('ORDERDATE'), 'yyyy-MM-dd'))
df.createTempView('dff')

In [0]:
%sql
select timestamp_col, cast(timestamp_col
 as timestamp) as cast_timestamp_col, quarter(timestamp_col), date_format(timestamp_col
, 'yyyy-MM-dd'), count(*) over () from dff

In [0]:
from pyspark.sql.functions import spark_partition_id
print(df.select(spark_partition_id()).distinct().count())